In [1]:
%pip install pymongo
!pip install transformers torch pandas scikit-learn

Note: you may need to restart the kernel to use updated packages.


In [2]:
pip install flask

Note: you may need to restart the kernel to use updated packages.


In [1]:
import os
from flask import Flask, request, jsonify
from pymongo import MongoClient
from bson import ObjectId
import pandas as pd

In [2]:
def connect_to_mongodb():
    try:
        mongo_uri = os.getenv('MONGODB_URI')
        if not mongo_uri:
            raise ValueError('MONGODB_URI environment variable is not set')
        client = MongoClient(mongo_uri)
        db = client["Student_Management_System"]
        collection = db["Students_Data"]
        
        # Test the connection by fetching one document
        if collection.find_one() is not None:
            print("✅ MongoDB connection successful and data is accessible!")
        else:
            print("✅ MongoDB connection successful, but the collection is empty.")
        
        return collection
    
    except Exception as e:
        print(f"❌ MongoDB connection failed: {e}")
        return None
    # Connect
collection = connect_to_mongodb()

✅ MongoDB connection successful and data is accessible!


In [3]:
app = Flask(__name__)

user = {}

#=============================CREATE USER===================================
@app.route("/create-user", methods = ["POST"]) 
def create_user():
    data = request.get_json()
    user_id = data.get("id")
    if not user_id:
        return jsonify({"error": "User ID is required"}), 400
        
    result = collection.insert_one(data)
    return jsonify({
        "message": "User created successfully",
        "inserted_id": str(result.inserted_id)
    }), 201

#=============================GETTING USER===================================
@app.route("/get-user", methods = ["GET"]) 
def get_user():
    user_id = request.args.get("id")
    if not user_id:
        return jsonify({"error": "User ID is required"}), 400
    
    user = collection.find_one({"id": user_id})
    if user:
        user["_id"] = str(user["_id"])  
        return jsonify({"user": user}), 200
    return jsonify({"error": "User not found"}), 404

#=============================UPDATING USER===================================
@app.route("/update-user", methods = ["PUT"])
def update_user():
    data = request.get_json()
    user_id = data.get("id")
    if not user_id:
        return jsonify({"error": "User ID is required"}), 400

    result = collection.update_one({"id": user_id}, {"$set": data})
    if result.matched_count > 0:
        updated_user = collection.find_one({"id": user_id})
        updated_user["_id"] = str(updated_user["_id"])
        return jsonify({"message": "User updated successfully", "user": updated_user}), 200
    return jsonify({"error": "User not found"}), 404

#=============================DELETE USER===================================
@app.route("/delete-user", methods = ["DELETE"])
def delete_user():
    user_id = request.args.get("id")
    if not user_id:
        return jsonify({"error": "User ID is required"}), 400

    result = collection.delete_one({"id": user_id})
    if result.deleted_count > 0:
        return jsonify({"message": "User deleted successfully"}), 204
    return jsonify({"error": "User not found"}), 404   

In [4]:
if __name__ == "__main__":
    app.run(debug=True, use_reloader=False)

 * Serving Flask app '__main__'
 * Debug mode: on


 * Running on http://127.0.0.1:5000
Press CTRL+C to quit
127.0.0.1 - - [13/Aug/2025 12:30:27] "PUT /delete-user?id=2 HTTP/1.1" 405 -
127.0.0.1 - - [13/Aug/2025 12:30:50] "PUT /delete-user?id=3 HTTP/1.1" 405 -
127.0.0.1 - - [13/Aug/2025 12:31:06] "PUT /update-user?id=3 HTTP/1.1" 200 -
127.0.0.1 - - [13/Aug/2025 12:31:41] "PUT /update-user?id=3 HTTP/1.1" 200 -
